# SecurePay AI - Local Evaluation on AMD AI Notebooks
This notebook was designed for the **AMD Developer Cloud (ROCm 7.2 + vLLM)** environment. We utilized the AMD Notebooks during the hackathon to evaluate inference latencies, prototype our Chain-of-Thought risk scoring prompts, and ensure compatibility with Google Gemma 3 on AMD MI300X accelerators before moving to the Fireworks Serverless API for our production frontend.

### 🔑 Hugging Face Authentication (Optional)
Google's Gemma models are gated on Hugging Face. To load the Gemma 3 model, make sure you have accepted the terms on the [Hugging Face model page](https://huggingface.co/google/gemma-3-4b-it).

If you have a Hugging Face token, you can run `huggingface-cli login` in the terminal or uncomment the cell below to authenticate. If no token is provided, the notebook will automatically fall back to the open, non-gated **Qwen 2.5 7B Instruct** model so you can evaluate the ROCm+vLLM pipeline without interruptions.

In [1]:
# # Optional: Login to Hugging Face to load gated Gemma models
# from huggingface_hub import login
# login("your_hugging_face_token_here")

In [2]:
import torch
from vllm import LLM, SamplingParams

print(f"PyTorch version: {torch.__version__}")
print(f"ROCm available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device name: {torch.cuda.get_device_name(0)}")

/opt/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch version: 2.9.1+gitff65f5b
ROCm available: True
Device name: 


In [3]:
# Initialize vLLM engine optimized for AMD MI300X
try:
    print("Attempting to load Google Gemma 3 model (google/gemma-3-4b-it)...\n")
    llm = LLM(model="google/gemma-3-4b-it", trust_remote_code=True, tensor_parallel_size=1)
except Exception as e:
    print(f"\nGemma 3 initialization failed: {e}\n")
    print("Falling back to non-gated open Qwen 2.5 model (Qwen/Qwen2.5-7B-Instruct)...\n")
    llm = LLM(model="Qwen/Qwen2.5-7B-Instruct", trust_remote_code=True, tensor_parallel_size=1)

sampling_params = SamplingParams(temperature=0.1, max_tokens=512)

Attempting to load Google Gemma 3 model (google/gemma-3-4b-it)...

INFO 07-13 03:16:30 [utils.py:223] non-default args: {'trust_remote_code': True, 'disable_log_stats': True, 'model': 'google/gemma-3-4b-it'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.



Gemma 3 initialization failed: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/google/gemma-3-4b-it.
401 Client Error. (Request ID: Root=1-6a54588f-7f2520f846a586cb39f76920;bd39fff6-262c-4227-93b2-2e57b621499d)

Cannot access gated repo for url http://134.199.133.77/google/gemma-3-4b-it/resolve/main/config.json.
Access to model google/gemma-3-4b-it is restricted. You must have access to it and be authenticated to access it. Please log in.

Falling back to non-gated open Qwen 2.5 model (Qwen/Qwen2.5-7B-Instruct)...

INFO 07-13 03:16:31 [utils.py:223] non-default args: {'trust_remote_code': True, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-7B-Instruct'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 07-13 03:16:34 [model.py:529] Resolved architecture: Qwen2ForCausalLM
INFO 07-13 03:16:34 [model.py:1549] Using max model len 32768


...

INFO 07-13 03:16:34 [scheduler.py:224] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 07-13 03:16:34 [vllm.py:689] Asynchronous scheduling is enabled.
WARNING 07-13 03:16:38 [system_utils.py:140] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
(EngineCore_DP0 pid=1800) INFO 07-13 03:16:44 [core.py:97] Initializing a V1 LLM engine (v0.16.1.dev0+g89a77b108.d20260318) with config: model='Qwen/Qwen2.5-7B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-7B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=32768, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=True, quantization=No

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  25% Completed | 1/4 [00:01<00:03,  1.32s/it]
Loading safetensors checkpoint shards:  50% Completed | 2/4 [00:02<00:02,  1.45s/it]
Loading safetensors checkpoint shards:  75% Completed | 3/4 [00:04<00:01,  1.49s/it]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:05<00:00,  1.47s/it]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:05<00:00,  1.46s/it]
(EngineCore_DP0 pid=1800) 


(EngineCore_DP0 pid=1800) INFO 07-13 03:16:55 [default_loader.py:293] Loading weights took 6.03 seconds
(EngineCore_DP0 pid=1800) INFO 07-13 03:16:56 [gpu_model_runner.py:4221] Model loading took 14.35 GiB memory and 9.163621 seconds
(EngineCore_DP0 pid=1800) INFO 07-13 03:16:59 [backends.py:916] Using cache directory: /root/.cache/vllm/torch_compile_cache/ca6001b4b6/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=1800) INFO 07-13 03:16:59 [backends.py:976] Dynamo bytecode transform time: 3.50 s
(EngineCore_DP0 pid=1800) INFO 07-13 03:17:03 [backends.py:267] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 2.324 s
(EngineCore_DP0 pid=1800) INFO 07-13 03:17:03 [monitor.py:34] torch.compile takes 5.82 s in total
(EngineCore_DP0 pid=1800) INFO 07-13 03:17:05 [gpu_worker.py:373] Available KV cache memory: 27.02 GiB
(EngineCore_DP0 pid=1800) INFO 07-13 03:17:05 [kv_cache_utils.py:1307] GPU KV cache size: 506,000 tokens
(EngineCore_DP0 pid=1

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:04<00:00, 10.89it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:05<00:00,  6.59it/s]


(EngineCore_DP0 pid=1800) INFO 07-13 03:17:16 [gpu_model_runner.py:5246] Graph capturing finished in 11 secs, took 3.22 GiB
(EngineCore_DP0 pid=1800) INFO 07-13 03:17:16 [core.py:278] init engine (profile, create kv cache, warmup model) took 20.14 seconds
(EngineCore_DP0 pid=1800) INFO 07-13 03:17:18 [vllm.py:689] Asynchronous scheduling is enabled.
INFO 07-13 03:17:18 [llm.py:355] Supported tasks: ['generate']


In [4]:
# Test our core Risk Analysis Prompt
system_prompt = "You are an elite payment security AI running on AMD hardware. Evaluate risk."
user_prompt = """
Evaluate the following transaction:
Amount: 15000 PKR
Merchant: Unknown Electronics Store
Device: New Device (Unrecognized)
Location: IP mismatch with billing zip
"""

formatted_prompt = f"<|system|>{system_prompt}<|user|>{user_prompt}<|assistant|>"
outputs = llm.generate([formatted_prompt], sampling_params)

for output in outputs:
    print(f"\n--- AI RISK ANALYSIS ---\n{output.outputs[0].text}")

Processed prompts: 100%|██████████| 1/1 [00:10<00:00, 10.53s/it, est. speed input: 6.37 toks/s, output: 31.54 toks/s]


--- AI RISK ANALYSIS ---
Based on the provided information, this transaction appears to be at a higher risk for potential fraud. Here are the key factors that contribute to this assessment:

1. **Merchant: Unknown Electronics Store** - Transactions with unknown or unfamiliar merchants are often flagged for additional scrutiny due to the lack of a known reputation and history.

2. **Device: New Device (Unrecognized)** - A new or unrecognized device can indicate that the transaction might be coming from a device that has not been previously associated with the account, which could be a sign of a compromised device or a new fraudulent attempt.

3. **Location: IP Mismatch with Billing Zip** - An IP address that does not match the billing zip code can suggest that the transaction is being made from a different location than where the customer is typically billed. This could be a red flag, as it might indicate that the transaction is being made from a different geographical area, possibly t